# mine-tuning-model

## 필수 라이브러리 설치

In [1]:
!pip install --upgrade numpy chromadb langchain langchain-community langchain-openai sentence-transformers transformers datasets gradio langgraph -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip freeze > /content/drive/MyDrive/Colab Notebooks/Github/mine-tuning-model/requirements.txt

## 허깅페이스에서 데이터 가져오기

In [ ]:
from datasets import load_dataset

ds = load_dataset("lparkourer10/minecraft-wiki")
print(ds)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# 샘플 3개 출력
for i in range(3):
    print(f"=== 샘플 {i+1} ===")
    print(f"URL   : {ds['train']['url'][i]}")
    print(f"Q     : {ds['train']['question'][i]}")
    print(f"A     : {ds['train']['answer'][i][:200]}")  # 앞 200자만
    print()

=== 샘플 1 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : What are the main differences between the original Minecraft gameplay and its current version in Windows 10 Edition?
A     : The main difference between the original Minecraft gameplay and its current version in Windows 10 Edition is the addition of new features and game modes, which have been implemented through Microsoft'

=== 샘플 2 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : Is the Creative mode in Minecraft really as easy to understand and use as it is portrayed in the summary?
A     : No, the creative mode in Minecraft is not entirely easy to understand and use as portrayed in the summary. In fact, some aspects of creative mode can be challenging for new players to grasp. For examp

=== 샘플 3 ===
URL   : https://minecraft.wiki/w/Minecraft_(franchise)
Q     : How does the addition of new biomes, mobs, and game modes in the recent update enhance the overall gameplay experience for player

# 데이터 임베딩


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print('[입베딩 성공]')
print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")

KeyboardInterrupt: 

## Chroma DB 구축

In [ ]:
import chromadb

# ChromaDB 초기화
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="minecraft_rag")

# 데이터 5000개 테스트
sample_size = 5000
answers   = ds["train"]["answer"][:sample_size]
questions = ds["train"]["question"][:sample_size]
urls      = ds["train"]["url"][:sample_size]

# 임베딩 생성 및 저장 (배치 처리)batch_size = 500
for i in range(0, sample_size, batch_size):
    batch_answers = answers[i:i+batch_size]
    embeddings = embedding_model.encode(batch_answers, show_progress_bar=True).tolist()

    collection.add(
        documents=batch_answers,
        embeddings=embeddings,
        metadatas=[{"question": q, "url": u} for q, u in zip(questions[i:i+batch_size], urls[i:i+batch_size])],
        ids=[str(j) for j in range(i, i+len(batch_answers))]
    )
    print(f" {min(i+batch_size, sample_size)}/{sample_size} 완료")

print(f"\n{collection.count()}개 저장됨")

AttributeError: `np.float_` was removed in the NumPy 2.0 release. Use `np.float64` instead.